In [0]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

## Load raw data

In [0]:
df = pd.read_csv("/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/raw/leads.csv")

df.shape

## Drop unnecessary columns

In [0]:
df = pd.DataFrame(df)
df = df.drop(['lead_id', 'company_name', 'created_date', 'job_title'], axis=1)

df.shape

## Train test split: Stratified Split

In [0]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=['converted'])
y = df['converted']

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Confirm shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Confirm stratification worked
print(f"\ny_train distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\ny_test distribution:\n{y_test.value_counts(normalize=True)}")

## Handle missing values

In [0]:
missing_values = pd.isnull(df).sum()
missing_values

In [0]:
median_days = X_train['days_since_last_contact'].median()
X_train['days_since_last_contact'] = X_train['days_since_last_contact'].fillna(median_days)
X_test['days_since_last_contact'] = X_test['days_since_last_contact'].fillna(median_days)

X_train['budget_indicated'] = X_train['budget_indicated'].fillna(0)
X_test['budget_indicated'] = X_test['budget_indicated'].fillna(0)

X_train['ad_platform'] = X_train['ad_platform'].fillna('unknown')
X_test['ad_platform'] = X_test['ad_platform'].fillna('unknown')

print(df.isnull().sum())


In [0]:
print(df.isnull().sum()[df.isnull().sum() > 0])

## Build Column Transformer

In [0]:
robust_cols = ['lead_age_days', 'email_opens', 'website_visits', 'content_downloads', 'days_since_last_contact', 'ad_clicks', 'num_contacts']

oe_cols = ['company_size', 'funnel_stage', 'preferred_contact_time']

ohe_cols = ['business_type', 'industry', 'lead_source', 'ad_platform']

preprocessor = ColumnTransformer(
    transformers=[
        ('robust', RobustScaler(), robust_cols),
        ('oe', OrdinalEncoder(categories=[
            ['Small', 'Medium', 'Large'],
            ['Awareness', 'Consideration', 'Decision'],
            ['Morning', 'Afternoon', 'Evening']
        ]), oe_cols),
        ('ohe', OneHotEncoder(handle_unknown='ignore'), ohe_cols)
    ],
    remainder='passthrough'
)

## Build ML Pipeline

In [0]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=10000))
])

In [0]:
pipeline.fit(X_train, y_train)

In [0]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

In [0]:
# Cell 1 — fit pipeline
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        class_weight={0: 1, 1: 10},
        random_state=42,
        n_estimators=100
    ))
])

pipeline_rf.fit(X_train, y_train)

# Cell 2 — get probabilities
y_prob_rf = pipeline_rf.predict_proba(X_test)[:, 1]

# Cell 3 — adjust threshold and evaluate
threshold = 0.15
y_pred_rf_adjusted = (y_prob_rf >= threshold).astype(int)

print(classification_report(y_test, y_pred_rf_adjusted))

In [0]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Get probabilities for both models
y_prob_lr = pipeline.predict_proba(X_test)[:, 1]
y_prob_rf = pipeline_rf.predict_proba(X_test)[:, 1]

# AUC Scores
auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"Logistic Regression AUC: {auc_lr:.3f}")
print(f"Random Forest AUC:       {auc_rf:.3f}")

# ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing (AUC = 0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Logistic Regression vs Random Forest')
plt.legend()
plt.tight_layout()
plt.show()

In [0]:

# Get feature names after preprocessing
feature_names = (
    pipeline_rf.named_steps['preprocessor']
    .get_feature_names_out()
)

# Get feature importances from Random Forest
importances = (
    pipeline_rf.named_steps['model']
    .feature_importances_
)

# Create dataframe and sort
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).head(15)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(
    feature_importance_df['feature'],
    feature_importance_df['importance']
)
plt.xlabel('Importance')
plt.title('Top 15 Most Important Features — Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(feature_importance_df)

In [0]:
# Get conversion probabilities for all leads
df_scoring = pd.read_csv("/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/raw/leads.csv")

# Apply same cleaning steps
df_scoring = df_scoring.drop(columns=['lead_id', 'job_title', 'company_name', 
                                       'created_date'])
df_scoring['budget_indicated'] = df_scoring['budget_indicated'].fillna(0)
df_scoring['ad_platform'] = df_scoring['ad_platform'].fillna('Unknown')

median_days = X_train['days_since_last_contact'].median()
df_scoring['days_since_last_contact'] = df_scoring['days_since_last_contact'].fillna(median_days)

# Separate features
X_scoring = df_scoring.drop(columns=['converted'])

# Get probabilities using best model
df_scoring['conversion_probability'] = pipeline_rf.predict_proba(X_scoring)[:, 1]

# Create lead score 0-100
df_scoring['lead_score'] = (df_scoring['conversion_probability'] * 100).round(1)

# Rank leads
df_scoring = df_scoring.sort_values('lead_score', ascending=False).reset_index(drop=True)
df_scoring['rank'] = df_scoring.index + 1

# Show top 20 leads
print(df_scoring[['rank', 'lead_score', 'conversion_probability', 'converted']].head(20))

# Save to outputs
df_scoring.to_csv("/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/outputs/scored_leads.csv", index=False)
print("\nScored leads saved successfully")